# 🏠 Feature Engineering Pipeline for Property DatasetThis notebook demonstrates data cleaning, transformation, and feature engineering steps to prepare a housing dataset for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('test.csv')
print(df.shape)
df.head()

## 1. Data Cleaning

In [ ]:
data = df.copy()
drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
data.drop(columns=drop_cols, inplace=True)

none_fill = [
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'FireplaceQu',
    'MasVnrType'
]
for col in none_fill:
    if col in data.columns:
        data[col] = data[col].fillna('None')

# LotFrontage — median by neighborhood
data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Fill numeric and categorical missing values
for col in data.select_dtypes(include=['int64','float64']).columns:
    data[col] = data[col].fillna(data[col].median())
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].fillna(data[col].mode()[0])

## 2. Feature Engineering

In [ ]:
data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
data['TotalBath'] = data['FullBath'] + 0.5*data['HalfBath'] + data['BsmtFullBath'] + 0.5*data['BsmtHalfBath']
data['Age'] = data['YrSold'] - data['YearBuilt']
data['RemodAge'] = data['YrSold'] - data['YearRemodAdd']
data.drop(columns=['Id'], inplace=True)

## 3. Encoding Categorical Data

In [ ]:
quality_map = { 'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0 }
ordinal_features = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
for col in ordinal_features:
    if col in data.columns:
        data[col] = data[col].map(quality_map).fillna(0).astype(int)

data = pd.get_dummies(data, columns=data.select_dtypes(include='object').columns, drop_first=True)

## 4. Handle Outliers and Scale Numeric Data

In [ ]:
num_cols = data.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    lower, upper = data[col].quantile(0.01), data[col].quantile(0.99)
    data[col] = np.clip(data[col], lower, upper)

scaler = RobustScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])

## 5. Final Checks and Export

In [ ]:
print('Shape:', data.shape)
print('Missing values:', data.isnull().sum().sum())
data.to_csv('test_preprocessed.csv', index=False)
data.head()